# Stock Price Graphs — All Days
Plots **trade price** over timestamp for all three trading days on a single graph per product.

- 🪨 **Ash-Coated Osmium** (`ASH_COATED_OSMIUM`)
- 🌿 **Intarian Pepper Root** (`INTARIAN_PEPPER_ROOT`)

Each day gets its own colour; timestamps are offset sequentially so Day −2, −1, 0 appear left→right.  
Expected files: `trades_round_1_day_-2.csv`, `trades_round_1_day_-1.csv`, `trades_round_1_day_0.csv`

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# ── Configuration ─────────────────────────────────────────────────────────────
DAYS = [-1, 0, 1]
DAY_COLORS = {
    1: "#4C72B0",   # blue
    -1: "#DD8452",   # orange
     0: "#55A868",   # green
}

PRODUCT_OSMIUM = "ASH_COATED_OSMIUM"
PRODUCT_PEPPER = "INTARIAN_PEPPER_ROOT"
# ─────────────────────────────────────────────────────────────────────────────

frames = {}
for day in DAYS:
    fname = f"trades_round_2_day_{day}.csv"
    fpath = os.path.join(os.getcwd(), fname)
    if os.path.exists(fpath):
        frames[day] = pd.read_csv(fpath, sep=";")
        print(f"Day {day:+d} -> {len(frames[day]):,} rows loaded from '{fname}'")
    else:
        frames[day] = pd.DataFrame(columns=["symbol", "timestamp", "price", "quantity"])
        print(f"Day {day:+d} -> FILE NOT FOUND: '{fname}'  (skipped)")

Day -1 -> FILE NOT FOUND: 'trades_round_2_day_-1.csv'  (skipped)
Day +0 -> FILE NOT FOUND: 'trades_round_2_day_0.csv'  (skipped)
Day +1 -> FILE NOT FOUND: 'trades_round_2_day_1.csv'  (skipped)


In [ ]:
# ── Split by product ──────────────────────────────────────────────────────────
product_data = {PRODUCT_OSMIUM: {}, PRODUCT_PEPPER: {}}

for day, df in frames.items():
    for prod in [PRODUCT_OSMIUM, PRODUCT_PEPPER]:
        subset = df[df["symbol"] == prod].sort_values("timestamp").reset_index(drop=True)
        product_data[prod][day] = subset

# ── Compute cumulative timestamp offsets so days appear sequentially ──────────
# offset[day] = total timestamps of all earlier days
# Uses the max timestamp across both products for that day as the day's span.
day_offset = {}
cumulative = 0
for day in DAYS:
    day_offset[day] = cumulative
    max_ts = 0
    for prod in [PRODUCT_OSMIUM, PRODUCT_PEPPER]:
        d = product_data[prod][day]
        if not d.empty:
            max_ts = max(max_ts, d["timestamp"].max())
    cumulative += max_ts + 1  # +1 so days don't share a boundary tick

print("Timestamp offsets:")
for day in DAYS:
    print(f"  Day {day:+d} offset: {day_offset[day]:,}")

print("\nTrades per product per day:")
print(f"{'Day':>6}  {'Osmium':>10}  {'Pepper':>10}")
for day in DAYS:
    osm_n = len(product_data[PRODUCT_OSMIUM][day])
    pep_n = len(product_data[PRODUCT_PEPPER][day])
    print(f"  {day:>+3}    {osm_n:>8,}    {pep_n:>8,}")

Timestamp offsets:
  Day -1 offset: 0
  Day +0 offset: 997,001
  Day +1 offset: 1,992,202

Trades per product per day:
   Day      Osmium      Pepper
   -1         459         331
   +0         471         332
   +1         465         333


In [1]:
# The height_ratios=[1, 3] makes the bottom plot (Pepper) 3 times taller than the top plot
fig, axes = plt.subplots(2, 1, figsize=(16, 12), constrained_layout=True, gridspec_kw={'height_ratios': [1, 3]})
fig.suptitle("Trade Price by Timestamp  —  Days -2, -1, 0  (offset sequentially)",
             fontsize=15, fontweight="bold")

def plot_product(ax, product, title):
    has_data = False

    # Day boundary vertical lines
    for day in DAYS[1:]:
        ax.axvline(x=day_offset[day], color="grey", linewidth=0.8,
                   linestyle="--", alpha=0.5, zorder=1)

    for day in DAYS:
        data  = product_data[product][day]
        color = DAY_COLORS[day]
        offset = day_offset[day]
        if data.empty:
            continue
        has_data = True
        ts     = data["timestamp"] + offset
        sizes  = data["quantity"] * 4 if "quantity" in data.columns else 20
        label  = f"Day {day:+d}  (open {data['price'].iloc[0]:,.1f}  close {data['price'].iloc[-1]:,.1f})"
        ax.scatter(ts, data["price"],
                   color=color, s=sizes, alpha=0.7, zorder=3, label=label)
        ax.plot(ts, data["price"],
                color=color, linewidth=0.9, alpha=0.4)

        # Annotate high and low for each day
        idx_max = data["price"].idxmax()
        idx_min = data["price"].idxmin()
        for idx, lbl, va in [(idx_max, "H", "bottom"), (idx_min, "L", "top")]:
            ax.annotate(
                f"{lbl}{day:+d} {data.loc[idx, 'price']:,.0f}",
                xy=(data.loc[idx, "timestamp"] + offset, data.loc[idx, "price"]),
                xytext=(6, 6 if va == "bottom" else -6),
                textcoords="offset points",
                fontsize=7, color=color, va=va,
                arrowprops=dict(arrowstyle="->", color=color, lw=0.7)
            )

    # Day labels on x-axis
    tick_positions = [day_offset[day] for day in DAYS]
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_positions)
    ax2.set_xticklabels([f"Day {d:+d}" for d in DAYS], fontsize=9)
    ax2.tick_params(length=0)

    if not has_data:
        ax.text(0.5, 0.5, "No data for any day", transform=ax.transAxes,
                ha="center", va="center", fontsize=12, color="grey")

    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.set_xlabel("Timestamp (offset per day)", fontsize=10)
    ax.set_ylabel("Trade Price", fontsize=10)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.1f}"))
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=9, loc="best")

plot_product(axes[0], PRODUCT_OSMIUM, "Ash-Coated Osmium  (ASH_COATED_OSMIUM)")
plot_product(axes[1], PRODUCT_PEPPER, "Intarian Pepper Root  (INTARIAN_PEPPER_ROOT)")

plt.savefig("all_days_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graph saved as  all_days_graph.png")

NameError: name 'plt' is not defined

In [9]:
for prod_name, prod_key in [("Ash-Coated Osmium", PRODUCT_OSMIUM),]:
    print(f"\n{'='*54}")
    print(f"  {prod_name}")
    print(f"{'='*54}")
    print(f"  {'Metric':<10}  {'Day -1':>12}  {'Day -0':>12}  {'Day  1':>12}")
    print(f"  {'-'*10}  {'-'*12}  {'-'*12}  {'-'*12}")

    metrics = ["Open", "Close", "High", "Low", "Mean", "Std",
               "Trades", "Volume", "Chg", "Chg %"]
    rows = {m: [] for m in metrics}

    for day in DAYS:
        data = product_data[prod_key][day]
        if data.empty:
            for key in metrics:
                rows[key].append("          --")
            continue
        s   = data["price"]
        chg = s.iloc[-1] - s.iloc[0]
        pct = chg / s.iloc[0] * 100
        rows["Open"].append(f"{s.iloc[0]:>12,.2f}")
        rows["Close"].append(f"{s.iloc[-1]:>12,.2f}")
        rows["High"].append(f"{s.max():>12,.2f}")
        rows["Low"].append(f"{s.min():>12,.2f}")
        rows["Mean"].append(f"{s.mean():>12,.2f}")
        rows["Std"].append(f"{s.std():>12,.2f}")
        rows["Trades"].append(f"{len(data):>12,}")
        rows["Volume"].append(f"{data['quantity'].sum():>12,}")
        rows["Chg"].append(f"{chg:>+12,.2f}")
        rows["Chg %"].append(f"{pct:>+11.2f}%")

    for metric in metrics:
        print(f"  {metric:<10}  {'  '.join(rows[metric])}")


  Ash-Coated Osmium
  Metric            Day -1        Day -0        Day  1
  ----------  ------------  ------------  ------------
  Open            9,982.00      9,995.00     10,000.00
  Close           9,992.00     10,016.00      9,985.00
  High           10,019.00     10,020.00     10,018.00
  Low             9,982.00      9,979.00      9,980.00
  Mean           10,000.88     10,000.98     10,000.09
  Std                 8.97          9.83          9.22
  Trades               459           471           465
  Volume             2,348         2,404         2,375
  Chg               +10.00        +21.00        -15.00
  Chg %             +0.10%        +0.21%        -0.15%


In [10]:
for prod_name, prod_key in [
                             ("Intarian Pepper Root", PRODUCT_PEPPER)]:
    print(f"\n{'='*54}")
    print(f"  {prod_name}")
    print(f"{'='*54}")
    print(f"  {'Metric':<10}  {'Day -1':>12}  {'Day  0':>12}  {'Day  1':>12}")
    print(f"  {'-'*10}  {'-'*12}  {'-'*12}  {'-'*12}")

    metrics = ["Open", "Close", "High", "Low", "Mean", "Std",
               "Trades", "Volume", "Chg", "Chg %"]
    rows = {m: [] for m in metrics}

    for day in DAYS:
        data = product_data[prod_key][day]
        if data.empty:
            for key in metrics:
                rows[key].append("          --")
            continue
        s   = data["price"]
        chg = s.iloc[-1] - s.iloc[0]
        pct = chg / s.iloc[0] * 100
        rows["Open"].append(f"{s.iloc[0]:>12,.2f}")
        rows["Close"].append(f"{s.iloc[-1]:>12,.2f}")
        rows["High"].append(f"{s.max():>12,.2f}")
        rows["Low"].append(f"{s.min():>12,.2f}")
        rows["Mean"].append(f"{s.mean():>12,.2f}")
        rows["Std"].append(f"{s.std():>12,.2f}")
        rows["Trades"].append(f"{len(data):>12,}")
        rows["Volume"].append(f"{data['quantity'].sum():>12,}")
        rows["Chg"].append(f"{chg:>+12,.2f}")
        rows["Chg %"].append(f"{pct:>+11.2f}%")

    for metric in metrics:
        print(f"  {metric:<10}  {'  '.join(rows[metric])}")


  Intarian Pepper Root
  Metric            Day -1        Day  0        Day  1
  ----------  ------------  ------------  ------------
  Open           11,010.00     12,011.00     13,011.00
  Close          11,998.00     12,987.00     13,999.00
  High           11,998.00     12,987.00     13,999.00
  Low            10,996.00     11,998.00     12,998.00
  Mean           11,540.49     12,535.24     13,526.27
  Std               284.99        287.63        289.88
  Trades               331           332           333
  Volume             1,669         1,671         1,693
  Chg              +988.00       +976.00       +988.00
  Chg %             +8.97%        +8.13%        +7.59%
